## 1. Install and import libraries


In [ ]:
# If needed, uncomment and run:
# %pip install -q pandas numpy matplotlib seaborn scikit-learn wordcloud

import os
import re
import html
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
print("Libraries imported successfully.")

## 2. Load the dataset


In [ ]:
def load_sms_dataset():
    candidates = ["SMS.csv", "spam.csv", "SMSSpamCollection.csv", "sms_spam.csv"]
    csv_path = next((Path(name) for name in candidates if Path(name).exists()), None)

    if csv_path is None:
        try:
            from google.colab import files
            print("Upload your SMS spam dataset CSV file:")
            uploaded = files.upload()
            if not uploaded:
                raise ValueError("No file was uploaded.")
            csv_path = Path(next(iter(uploaded.keys())))
        except ImportError:
            raise FileNotFoundError(
                "Dataset CSV not found. Place spam.csv or SMS.csv in the notebook folder, "
                "or run this notebook in Google Colab and upload the file."
            )

    # The Kaggle file often contains extra unnamed columns and uses latin-1 encoding.
    raw = pd.read_csv(csv_path, encoding="latin-1")
    if {"v1", "v2"}.issubset(raw.columns):
        data = raw[["v1", "v2"]].rename(columns={"v1": "label", "v2": "message"})
    elif {"label", "message"}.issubset(raw.columns):
        data = raw[["label", "message"]].copy()
    else:
        raise ValueError(
            f"Could not identify label/message columns in {csv_path.name}. "
            f"Available columns: {list(raw.columns)}"
        )

    data["label"] = data["label"].astype(str).str.strip().str.lower()
    data["message"] = data["message"].fillna("").astype(str)
    data = data[data["label"].isin(["ham", "spam"])].copy()
    data = data[data["message"].str.strip().ne("")].copy()
    data = data.drop_duplicates(subset=["label", "message"]).reset_index(drop=True)
    if data.empty:
        raise ValueError("No valid ham/spam rows were found in the dataset.")
    print(f"Loaded: {csv_path}")
    return data

df = load_sms_dataset()
print("Dataset shape:", df.shape)
display(df.head())

## 3. Understand and validate the data


In [ ]:
print("Dataset information:")
df.info()
print("\nMissing values:\n", df.isna().sum())
print("\nLabel counts:\n", df["label"].value_counts())
print("\nClass proportions:\n", df["label"].value_counts(normalize=True).round(3))

assert set(df["label"].unique()).issubset({"ham", "spam"}), "Unexpected label found."
assert df["message"].str.strip().ne("").all(), "Empty messages remain."
print("\nData validation completed.")

## 4. Exploratory data analysis (EDA)


In [ ]:
eda_df = df.copy()
eda_df["character_count"] = eda_df["message"].str.len()
eda_df["word_count"] = eda_df["message"].str.split().str.len()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(data=eda_df, x="label", order=["ham", "spam"], ax=axes[0])
axes[0].set_title("Number of messages by class")
axes[0].set_xlabel("Class")

sns.histplot(data=eda_df, x="word_count", hue="label", bins=35, element="step", stat="count", common_norm=False, ax=axes[1])
axes[1].set_title("Message word counts")
axes[1].set_xlim(0, eda_df["word_count"].quantile(0.99))

sns.boxplot(data=eda_df, x="label", y="character_count", order=["ham", "spam"], showfliers=False, ax=axes[2])
axes[2].set_title("Message character counts")
plt.tight_layout()
plt.show()

print("Average word count by class:")
display(eda_df.groupby("label")[["word_count", "character_count"]].mean().round(2))

In [ ]:
# Optional word clouds. If wordcloud is not installed, skip this cell or install it.
try:
    from wordcloud import WordCloud
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, label in zip(axes, ["ham", "spam"]):
        text = " ".join(df.loc[df["label"] == label, "message"].tolist())
        if text.strip():
            cloud = WordCloud(width=800, height=400, background_color="white", collocations=False).generate(text)
            ax.imshow(cloud, interpolation="bilinear")
        ax.set_title(f"Most common words: {label.upper()}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("WordCloud is not installed. Install it with: %pip install wordcloud")

## 5. Text preprocessing


In [ ]:
def clean_text(text):
    """Normalize an SMS message while preserving useful word/number signals."""
    text = html.unescape(str(text)).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " URLTOKEN ", text)
    text = re.sub(r"\b[\w.+%-]+@[\w.-]+\.[a-z]{2,}\b", " EMAILTOKEN ", text)
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

df["clean_message"] = df["message"].apply(clean_text)
df = df[df["clean_message"].str.len() > 0].reset_index(drop=True)
display(df[["label", "message", "clean_message"]].head())

## 6. Split into training and test sets


In [ ]:
X = df["clean_message"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f"Training messages: {len(X_train):,}")
print(f"Test messages: {len(X_test):,}")
print("Training class proportions:\n", y_train.value_counts(normalize=True).round(3))
print("Test class proportions:\n", y_test.value_counts(normalize=True).round(3))

## 7. Build and train the baseline SVM model


In [ ]:
baseline_model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=10000, sublinear_tf=True)),
    ("svm", LinearSVC(class_weight="balanced", random_state=RANDOM_STATE))
])

baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)

print("Baseline test metrics")
print(f"Accuracy : {accuracy_score(y_test, baseline_pred):.4f}")
print(f"Precision: {precision_score(y_test, baseline_pred, pos_label='spam', zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test, baseline_pred, pos_label='spam', zero_division=0):.4f}")
print(f"F1-score : {f1_score(y_test, baseline_pred, pos_label='spam', zero_division=0):.4f}")
print("\nClassification report:\n")
print(classification_report(y_test, baseline_pred, labels=["ham", "spam"], zero_division=0))

## 8. Tune SVM hyperparameters


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
param_grid = {
    "tfidf__max_features": [5000, 10000],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "svm__C": [0.1, 1, 10]
}

grid = GridSearchCV(
    estimator=Pipeline([
        ("tfidf", TfidfVectorizer(sublinear_tf=True)),
        ("svm", LinearSVC(class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    refit=True
)
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print(f"Best mean CV macro-F1: {grid.best_score_:.4f}")
model = grid.best_estimator_

## 9. Final test-set evaluation


In [ ]:
test_pred = model.predict(X_test)
metrics = {
    "Accuracy": accuracy_score(y_test, test_pred),
    "Spam precision": precision_score(y_test, test_pred, pos_label="spam", zero_division=0),
    "Spam recall": recall_score(y_test, test_pred, pos_label="spam", zero_division=0),
    "Spam F1-score": f1_score(y_test, test_pred, pos_label="spam", zero_division=0),
    "Macro F1-score": f1_score(y_test, test_pred, average="macro", zero_division=0)
}
metrics_df = pd.DataFrame({"Metric": metrics.keys(), "Score": metrics.values()})
display(metrics_df.style.format({"Score": "{:.4f}"}))
print(classification_report(y_test, test_pred, labels=["ham", "spam"], target_names=["Ham (not spam)", "Spam"], zero_division=0))

cm = confusion_matrix(y_test, test_pred, labels=["ham", "spam"])
display_cm = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Ham", "Spam"])
display_cm.plot(cmap="Blues", values_format="d")
plt.title("SVM Confusion Matrix — Test Set")
plt.tight_layout()
plt.show()

## 10. Cross-validation score


In [ ]:
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1)
print("Fold macro-F1 scores:", np.round(cv_scores, 4))
print(f"Mean macro-F1: {cv_scores.mean():.4f}")
print(f"Standard deviation: {cv_scores.std():.4f}")

## 11. Inspect influential words (Linear SVM)


In [ ]:
vectorizer = model.named_steps["tfidf"]
svm = model.named_steps["svm"]
feature_names = np.array(vectorizer.get_feature_names_out())
classes = list(svm.classes_)

if len(classes) == 2:
    weights = svm.coef_[0]
    # In binary LinearSVC, positive weights favor classes_[1] and negative weights favor classes_[0].
    top_for_class_1 = feature_names[np.argsort(weights)[-15:][::-1]]
    top_for_class_0 = feature_names[np.argsort(weights)[:15]]
    print(f"Words/phrases associated with {classes[1]}:")
    print(", ".join(top_for_class_1))
    print(f"\nWords/phrases associated with {classes[0]}:")
    print(", ".join(top_for_class_0))

## 12. Predict new SMS messages


In [ ]:
def predict_sms(message, fitted_model=model):
    """Return predicted class and decision score for one SMS message."""
    cleaned = clean_text(message)
    if not cleaned:
        raise ValueError("Please enter a non-empty SMS message.")
    prediction = fitted_model.predict([cleaned])[0]
    score = float(fitted_model.decision_function([cleaned])[0])
    return {"message": message, "prediction": prediction.upper(), "decision_score": round(score, 4)}

examples = [
    "Congratulations! You have won a free iPhone. Click here now.",
    "Hi, are we meeting tomorrow at 3 PM?"
]
for example in examples:
    print(predict_sms(example))

In [ ]:
# Try your own message by changing the text below.
my_message = "URGENT! You have won a cash prize. Call now to claim."
print(predict_sms(my_message))

## 13. Save the trained model (optional)


In [ ]:
# Uncomment to save the model locally:
# import joblib
# joblib.dump(model, "sms_spam_svm.joblib")
# print("Saved as sms_spam_svm.joblib")

# To load it later:
# loaded_model = joblib.load("sms_spam_svm.joblib")